# 🚀 Derm-AI: Automated Multi-Dataset Fine-Tuning Pipeline
This pipeline automatically downloads, harmonizes, and merges **all 4** datasets you requested:
- **Fitzpatrick 17k** (SkinCAP version)
- **Derm1M** (HuggingFace)
- **SkinGPT-4** (HuggingFace)
- **DDI** (Diverse Dermatology Images via HF mirror)

It then trains a Vision Transformer (ViT) and downloads the final model.

In [ ]:
# 1. Install required packages
!pip install -q datasets transformers[torch] accelerate evaluate

In [ ]:
# 2. Authenticate with HuggingFace (Required for Gated Datasets like Derm1M)
from huggingface_hub import login
from google.colab import userdata

try:
    login(token=userdata.get('HF_TOKEN'))
    print('Successfully logged in via Colab Secrets!')
except:
    print('Please enter your HuggingFace token below:')
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# 3. Load & Harmonize All 4 Datasets
from datasets import load_dataset, concatenate_datasets, Dataset
import pandas as pd

all_datasets = []

def standardize_dataset(ds, label_col_candidates, default_label='Unknown'):
    # Standardize image column
    img_col = next((c for c in ds.column_names if 'img' in c.lower() or 'image' in c.lower()), None)
    if img_col and img_col != 'image':
        ds = ds.rename_column(img_col, 'image')
    
    # Standardize label column
    lbl_col = next((c for c in ds.column_names if c in label_col_candidates), None)
    if lbl_col:
        if lbl_col != 'label':
            ds = ds.rename_column(lbl_col, 'label')
    else:
        # If no label exists (like raw text/caption datasets), create a generic label
        ds = ds.add_column('label', [default_label] * len(ds))
    
    # Keep ONLY image and label to allow safe concatenation
    return ds.select_columns(['image', 'label'])

# --- 1. Fitzpatrick 17k ---
try:
    print('Downloading Fitzpatrick 17k...')
    ds_fitz = load_dataset('joshuachou/SkinCAP', split='train')
    ds_fitz = standardize_dataset(ds_fitz, ['condition', 'diagnosis', 'label'])
    all_datasets.append(ds_fitz)
    print(f'✅ Added {len(ds_fitz)} images from Fitzpatrick17k.')
except Exception as e:
    print(f'❌ Skipped Fitzpatrick17k: {e}')

# --- 2. Derm1M ---
try:
    print('Downloading Derm1M...')
    ds_derm1m = load_dataset('redlessone/Derm1M', split='train')
    ds_derm1m = standardize_dataset(ds_derm1m, ['label', 'condition_name', 'text'], 'Derm1M_Unspecified')
    all_datasets.append(ds_derm1m)
    print(f'✅ Added {len(ds_derm1m)} images from Derm1M.')
except Exception as e:
    print(f'❌ Skipped Derm1M (You may need to accept the dataset terms on HF first): {e}')

# --- 3. SkinGPT-4 ---
try:
    print('Downloading SkinGPT-4...')
    ds_skingpt = load_dataset('Jiaxi-Zhao/SkinGPT-4', split='train')
    ds_skingpt = standardize_dataset(ds_skingpt, ['label', 'diagnosis'], 'SkinGPT_Unspecified')
    all_datasets.append(ds_skingpt)
    print(f'✅ Added {len(ds_skingpt)} images from SkinGPT-4.')
except Exception as e:
    print(f'❌ Skipped SkinGPT-4: {e}')

# --- 4. DDI (Diverse Dermatology Images) ---
try:
    print('Downloading DDI...')
    ds_ddi = load_dataset('ddi_dataset', split='train') # Note: User may need to specify exact DDI repo name if private
    ds_ddi = standardize_dataset(ds_ddi, ['malignancy', 'diagnosis', 'label'])
    all_datasets.append(ds_ddi)
    print(f'✅ Added {len(ds_ddi)} images from DDI.')
except Exception as e:
    print(f'❌ Skipped DDI: {e}')

# --- Merge All Datasets ---
if not all_datasets:
    raise ValueError('No datasets were loaded successfully. Please check your HuggingFace token and access permissions.')

final_dataset = concatenate_datasets(all_datasets)
print(f'\n🚀 SUCCESS! Unified Master Dataset created with {len(final_dataset)} total images!')

# Convert string labels to integers
final_dataset = final_dataset.cast_column('label', ClassLabel(names=list(set(final_dataset['label']))))

unique_labels = final_dataset.features['label'].names
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}

# Split into Train and Validation (90/10 split)
final_dataset = final_dataset.train_test_split(test_size=0.1, seed=42)
train_ds = final_dataset['train']
val_ds = final_dataset['test']
print(f'Training on {len(train_ds)} images, Validating on {len(val_ds)} images. Total classes: {len(unique_labels)}')


In [ ]:
# 4. Preprocess Images for Vision Transformer (ViT)
from transformers import AutoImageProcessor, AutoModelForImageClassification
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor

model_name = 'google/vit-base-patch16-224-in21k'
processor = AutoImageProcessor.from_pretrained(model_name)

normalize = Normalize(mean=processor.image_mean, std=processor.image_std)
train_transforms = Compose([
    RandomResizedCrop(processor.size['height']),
    ToTensor(),
    normalize,
])

val_transforms = Compose([
    RandomResizedCrop(processor.size['height']),
    ToTensor(),
    normalize,
])

def preprocess_train(example_batch):
    example_batch['pixel_values'] = [
        train_transforms(image.convert('RGB')) for image in example_batch['image']
    ]
    return example_batch

def preprocess_val(example_batch):
    example_batch['pixel_values'] = [
        val_transforms(image.convert('RGB')) for image in example_batch['image']
    ]
    return example_batch

train_ds.set_transform(preprocess_train)
val_ds.set_transform(preprocess_val)

import torch
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['label'] for x in batch])
    }

In [ ]:
# 5. Initialize Model & Training Arguments
model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

metric = evaluate.load('accuracy')
def compute_metrics(p):
    return metric.compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids)

training_args = TrainingArguments(
    output_dir='./derm_ai_model_checkpoints',
    remove_unused_columns=False,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy'
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=processor,
    compute_metrics=compute_metrics,
)

In [ ]:
# 6. Start Fine-Tuning!
print('Starting training process. This may take several hours depending on the dataset size...')
trainer.train()

In [ ]:
# 7. Export and Download the Fine-Tuned Model
import shutil
from google.colab import files

save_path = './derm_ai_final_model'

print(f'Saving final model to {save_path}...')
trainer.save_model(save_path)
processor.save_pretrained(save_path)

print('Zipping the model for download...')
shutil.make_archive('derm_ai_final_model', 'zip', save_path)

print('Initiating download to your computer! Please leave the browser tab open.')
files.download('derm_ai_final_model.zip')